In [1]:
from google.colab import files
files.upload()

Saving kaggle.json to kaggle.json


{'kaggle.json': b'{"username":"amairanyfdez","key":"f7b5a4b47f6d2ac68c8352fb9a01b1e9"}'}

In [2]:
import os
os.makedirs(os.path.expanduser("~/.config/kaggle"), exist_ok=True)
os.rename("kaggle.json", os.path.expanduser("~/.config/kaggle/kaggle.json"))
os.chmod(os.path.expanduser("~/.config/kaggle/kaggle.json"), 0o600)
print("Kaggle API configurada correctamente")

Kaggle API configurada correctamente


In [3]:
import kagglehub

path = kagglehub.dataset_download("jeanmidev/smart-meters-in-london")
print("Dataset descargado en:", path)

100%|██████████| 1.17G/1.17G [00:14<00:00, 84.2MB/s]

Extracting files...


Dataset descargado en: /root/.cache/kagglehub/datasets/jeanmidev/smart-meters-in-london/versions/21


In [4]:
import os

for root, dirs, files in os.walk(path):
    level = root.replace(path, '').count(os.sep)
    indent = ' ' * 2 * level
    print(f"{indent}{os.path.basename(root)}/")
    subindent = ' ' * 2 * (level + 1)
    for file in files:
        print(f"{subindent}{file}")

21/
  acorn_details.csv
  uk_bank_holidays.csv
  weather_hourly_darksky.csv
  weather_daily_darksky.csv
  daily_dataset.csv
  darksky_parameters_documentation.html
  informations_households.csv
  halfhourly_dataset/
    halfhourly_dataset/
      block_35.csv
      block_30.csv
      block_105.csv
      block_66.csv
      block_88.csv
      block_48.csv
      block_19.csv
      block_74.csv
      block_21.csv
      block_72.csv
      block_39.csv
      block_84.csv
      block_67.csv
      block_64.csv
      block_95.csv
      block_54.csv
      block_56.csv
      block_29.csv
      block_16.csv
      block_76.csv
      block_103.csv
      block_59.csv
      block_10.csv
      block_50.csv
      block_70.csv
      block_32.csv
      block_49.csv
      block_7.csv
      block_63.csv
      block_93.csv
      block_108.csv
      block_83.csv
      block_18.csv
      block_79.csv
      block_43.csv
      block_20.csv
      block_98.csv
      block_81.csv
      block_102.csv
      block_111.

In [5]:
# SMART ENERGY ANALYTICS - Modelo de Regresión

import os
import glob
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.preprocessing import LabelEncoder
import xgboost as xgb
import warnings
warnings.filterwarnings('ignore')

print("Librerías importadas correctamente")

Librerías importadas correctamente


In [6]:
# Carga de datos

base_path = "/root/.cache/kagglehub/datasets/jeanmidev/smart-meters-in-london/versions/21"

# Carga de daily_dataset (todos los bloques)
print("Cargando daily_dataset...")
daily_path = os.path.join(base_path, "daily_dataset", "daily_dataset")
bloques = glob.glob(os.path.join(daily_path, "*.csv"))

df_list = []
for bloque in bloques:
    df_temp = pd.read_csv(bloque)
    df_list.append(df_temp)

df_daily = pd.concat(df_list, ignore_index=True)
print(f"daily_dataset cargado: {df_daily.shape[0]:,} filas, {df_daily.shape[1]} columnas")

# --- 2.2 Cargar clima diario ---
print("Cargando weather_daily_darksky...")
df_weather = pd.read_csv(os.path.join(base_path, "weather_daily_darksky.csv"))
print(f"Weather cargado: {df_weather.shape[0]:,} filas, {df_weather.shape[1]} columnas")

# --- 2.3 Cargar información de hogares ---
print("Cargando informations_households...")
df_households = pd.read_csv(os.path.join(base_path, "informations_households.csv"))
print(f"Households cargado: {df_households.shape[0]:,} filas, {df_households.shape[1]} columnas")

# Vista rápida
print("\n--- Primeras filas daily_dataset ---")
display(df_daily.head())

Cargando daily_dataset...
daily_dataset cargado: 3,510,433 filas, 9 columnas
Cargando weather_daily_darksky...
Weather cargado: 882 filas, 32 columnas
Cargando informations_households...
Households cargado: 5,566 filas, 5 columnas

--- Primeras filas daily_dataset ---


,LCLid,day,energy_median,energy_mean,energy_max,energy_count,energy_std,energy_sum,energy_min
0,MAC000028,2011-12-07,0.1270,0.122840,0.217,25,0.046093,3.071,0.053
1,MAC000028,2011-12-08,0.0990,0.110292,0.237,48,0.047222,5.294,0.054
2,MAC000028,2011-12-09,0.0890,0.105729,0.238,48,0.047683,5.075,0.053
3,MAC000028,2011-12-10,0.0875,0.113042,0.541,48,0.077243,5.426,0.046
4,MAC000028,2011-12-11,0.1030,0.107708,0.247,48,0.049081,5.170,0.052


In [7]:
# Limpieza y preparación

df_daily['day'] = pd.to_datetime(df_daily['day'])
df_weather['time'] = pd.to_datetime(df_weather['time'])

print("Nulos en daily_dataset:")
print(df_daily.isnull().sum())
print("\nNulos en weather:")
print(df_weather.isnull().sum()[df_weather.isnull().sum() > 0])

df_daily = df_daily.dropna(subset=['energy_sum', 'energy_mean'])
print(f"\n Filas después de limpieza: {df_daily.shape[0]:,}")

dias_por_hogar = df_daily.groupby('LCLid')['day'].count()
hogares_validos = dias_por_hogar[dias_por_hogar >= 300].index
df_daily = df_daily[df_daily['LCLid'].isin(hogares_validos)]
print(f" Hogares con cobertura suficiente: {df_daily['LCLid'].nunique():,}")

df = df_daily.merge(
    df_weather[['time', 'temperatureMax', 'temperatureMin', 'humidity',
                'windSpeed', 'uvIndex', 'precipType']],
    left_on='day',
    right_on='time',
    how='left'
).drop(columns='time')

print(f" Dataset con clima: {df.shape[0]:,} filas, {df.shape[1]} columnas")
print("\n--- Vista previa ---")
display(df.head())

Nulos en daily_dataset:
LCLid                0
day                  0
energy_median       30
energy_mean         30
energy_max          30
energy_count         0
energy_std       11331
energy_sum          30
energy_min          30
dtype: int64

Nulos en weather:
cloudCover     1
uvIndex        1
uvIndexTime    1
dtype: int64

 Filas después de limpieza: 3,510,403
 Hogares con cobertura suficiente: 5,446
 Dataset con clima: 3,484,756 filas, 15 columnas

--- Vista previa ---


,LCLid,day,energy_median,energy_mean,energy_max,energy_count,energy_std,energy_sum,energy_min,temperatureMax,temperatureMin,humidity,windSpeed,uvIndex,precipType
0,MAC000028,2011-12-07,0.1270,0.122840,0.217,25,0.046093,3.071,0.053,9.02,4.91,0.68,7.06,1.0,rain
1,MAC000028,2011-12-08,0.0990,0.110292,0.237,48,0.047222,5.294,0.054,12.89,4.27,0.81,7.01,1.0,rain
2,MAC000028,2011-12-09,0.0890,0.105729,0.238,48,0.047683,5.075,0.053,7.68,2.03,0.71,5.65,1.0,rain
3,MAC000028,2011-12-10,0.0875,0.113042,0.541,48,0.077243,5.426,0.046,6.08,-0.13,0.81,3.08,1.0,rain
4,MAC000028,2011-12-11,0.1030,0.107708,0.247,48,0.049081,5.170,0.052,8.59,2.48,0.88,3.94,1.0,rain


In [8]:
# * Feature engineering

df = df.sort_values(['LCLid', 'day']).reset_index(drop=True)

# Variables temporales
df['mes']         = df['day'].dt.month
df['dia_semana']  = df['day'].dt.dayofweek   # 0=Lunes, 6=Domingo
df['dia_año']     = df['day'].dt.dayofyear
df['es_fin_semana'] = (df['dia_semana'] >= 5).astype(int)
df['estacion'] = df['mes'].map({
    12: 'invierno', 1: 'invierno', 2: 'invierno',
    3: 'primavera', 4: 'primavera', 5: 'primavera',
    6: 'verano',    7: 'verano',    8: 'verano',
    9: 'otoño',    10: 'otoño',    11: 'otoño'
})

# Variables climáticas derivadas
df['temp_rango'] = df['temperatureMax'] - df['temperatureMin']
df['HDD'] = (15.5 - df['temperatureMin']).clip(lower=0)  # Heating Degree Days

# Lag features por hogar (consumo pasado)
print("Calculando lag features... (puede tardar un momento)")
df['lag_1d']  = df.groupby('LCLid')['energy_sum'].shift(1)
df['lag_7d']  = df.groupby('LCLid')['energy_sum'].shift(7)
df['lag_30d'] = df.groupby('LCLid')['energy_sum'].shift(30)

# Rolling averages por hogar
df['rolling_7d']  = df.groupby('LCLid')['energy_sum'].transform(
    lambda x: x.shift(1).rolling(7).mean())
df['rolling_30d'] = df.groupby('LCLid')['energy_sum'].transform(
    lambda x: x.shift(1).rolling(30).mean())

# Codificando variables categóricas
le = LabelEncoder()
df['estacion_cod'] = le.fit_transform(df['estacion'])
df['precipType'] = df['precipType'].fillna('none')
df['precipType_cod'] = le.fit_transform(df['precipType'])

# Eliminar filas con NaN generados por lags
df_model = df.dropna(subset=['lag_1d', 'lag_7d', 'lag_30d',
                              'rolling_7d', 'rolling_30d'])
print(f"Dataset final para modelado: {df_model.shape[0]:,} filas")
print(f"Features creadas: {df_model.shape[1]} columnas")
print("\nNuevas features:", ['mes','dia_semana','es_fin_semana','estacion_cod',
                             'temp_rango','HDD','lag_1d','lag_7d','lag_30d',
                             'rolling_7d','rolling_30d'])

Calculando lag features... (puede tardar un momento)
Dataset final para modelado: 3,321,376 filas
Features creadas: 29 columnas

Nuevas features: ['mes', 'dia_semana', 'es_fin_semana', 'estacion_cod', 'temp_rango', 'HDD', 'lag_1d', 'lag_7d', 'lag_30d', 'rolling_7d', 'rolling_30d']


In [9]:
# Preparando el modelado

# Definimos features y variable objetivo
features = [
    'energy_median', 'energy_mean', 'energy_max', 'energy_min',
    'mes', 'dia_semana', 'dia_año', 'es_fin_semana', 'estacion_cod',
    'temperatureMax', 'temperatureMin', 'temp_rango', 'HDD',
    'humidity', 'windSpeed', 'precipType_cod',
    'lag_1d', 'lag_7d', 'lag_30d',
    'rolling_7d', 'rolling_30d'
]

target = 'energy_sum'

X = df_model[features]
y = df_model[target]

# Train/Test Split (80/20) respetando orden temporal
# Usamos orden temporal para no filtrar información del futuro al pasado
split_idx = int(len(df_model) * 0.8)
X_train, X_test = X.iloc[:split_idx], X.iloc[split_idx:]
y_train, y_test = y.iloc[:split_idx], y.iloc[split_idx:]

print(f"Train: {X_train.shape[0]:,} filas ({X_train.shape[0]/len(X)*100:.0f}%)")
print(f"Test:  {X_test.shape[0]:,} filas ({X_test.shape[0]/len(X)*100:.0f}%)")
print(f"Features usadas: {len(features)}")
print(f"\nVariable objetivo → '{target}'")
print(f"  Media: {y.mean():.3f} kWh")
print(f"  Min:   {y.min():.3f} kWh")
print(f"  Max:   {y.max():.3f} kWh")

Train: 2,657,100 filas (80%)
Test:  664,276 filas (20%)
Features usadas: 21

Variable objetivo → 'energy_sum'
  Media: 10.140 kWh
  Min:   0.000 kWh
  Max:   332.556 kWh


In [ ]:
# Entrenamiento de modelos

from sklearn.ensemble import RandomForestRegressor
import xgboost as xgb
import time

# Random Forest
print("Entrenando Random Forest...")
t0 = time.time()
rf_model = RandomForestRegressor(
    n_estimators=100,
    max_depth=15,
    min_samples_leaf=10,
    n_jobs=-1,
    random_state=42
)
rf_model.fit(X_train, y_train)
t_rf = time.time() - t0
print(f" Random Forest entrenado en {t_rf:.1f} segundos")

# XGBoost
print("\nEntrenando XGBoost...")
t0 = time.time()
xgb_model = xgb.XGBRegressor(
    n_estimators=200,
    max_depth=8,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    n_jobs=-1,
    random_state=42,
    verbosity=0
)
xgb_model.fit(X_train, y_train)
t_xgb = time.time() - t0
print(f"XGBoost entrenado en {t_xgb:.1f} segundos")

print("\nAmbos modelos entrenados correctamente")

Entrenando Random Forest...
